# Part 2 - Modelling
## Chapter 9 - Hyper-Parameter Tuning with Cross-Validation

### Using the function getTestData from Chapter 8, form a synthetic dataset of 10,000 observations with 10 features, where 5 are informative and 5 are noise.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification


def get_test_data(n_features=40, n_informative=10, n_redundant=10, n_samples=10000):
    # generate a random dataset for a classification problem
    X, labels = make_classification(n_samples=n_samples, n_features=n_features,
                                    n_informative=n_informative, n_redundant=n_redundant, random_state=0,
                                    shuffle=False)
    indices = pd.date_range(end=pd.Timestamp.today(), periods=n_samples, freq='min').astype('int64') // 10**6
    X, labels = pd.DataFrame(X, index=indices), pd.Series(labels, index=indices).to_frame('bin')
    columns = [f'I_{i}' for i in range(n_informative)]
    columns += [f'R_{i}' for i in range(n_redundant)]
    columns += [f'N_{i}' for i in range(n_features - len(columns))]
    X.columns = columns
    labels['w'] = 1. / labels.shape[0]
    labels['t1'] = pd.Series(labels.index, index=labels.index)
    return X, labels

X, y = get_test_data(n_features=10, n_informative=5, n_redundant=0, n_samples=10000)

#### 9.1 (a) Use GridSearchCV on 10-fold CV to find the C, gamma optimal hyperparameters on a SVC with RBF kernel, where `param_grid={'C':[1E2,1E-1,1,10,100],'gamma':[1E-2,1E-1,1,10,100]}` and the scoring function is `neg_log_loss`.

In [2]:
import importlib
from afml.modelling import hyperparameter_tuning
importlib.reload(hyperparameter_tuning)
from afml.modelling.hyperparameter_tuning import tune_hyper_params
from sklearn.svm import SVC

search = tune_hyper_params(
    X,
    y['bin'],
    y['t1'],
    pipe_estimator=SVC(kernel='rbf'),
    param_grid={'C':[1E2,1E-1,1,10,100],'gamma':[1E-2,1E-1,1,10,100]},
    cv=3,
    bagging=None,
    rnd_search_iter=0,
    n_jobs=-1,
    pct_embargo=0,
)

ValueError: 
All the 75 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
75 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/yakirhadad/PycharmProjects/Advances-In-Financial-Machine-Learning/venv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/yakirhadad/PycharmProjects/Advances-In-Financial-Machine-Learning/venv/lib/python3.11/site-packages/sklearn/base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: BaseLibSVM.fit() got an unexpected keyword argument 'bagging'


#### 9.1 (b) How many nodes are there in the grid?
The number of nodes in the grid in a GridSearchCV context refers to the total number of parameter combinations that are evaluated during the grid search. This is calculated as the product of all the possible values for each parameter in the grid. In our case, we have 5 values for C and 5 values for gamma, so the total number of nodes is: 25

#### 9.1 (c) How many fits did it take to find the optimal solution?
The number of fits it takes to find the optimal solution depends on the number of parameter combinations and the number of cross-validation folds. The total number of fits is calculated as:

Total Fits = Number of Parameter Combinations × Number of Cross-Validation Folds
So in our case is 25 nodes X 3 folds = 75 fits 

#### 9.1 (d) How long did it take to find this solution?
17 seconds

#### 9.1 (e) How can you access the optimal result?
using best_params_ attribute of the GridSearchCV object

In [ ]:
search.best_params_

#### 9.1 (f) What is the CV score of the optimal parameter combination?

In [ ]:
search.best_score_

#### 9.1 (g) How can you pass sample weights to the SVC?
passing sample weights to svc is simply on `fit()` method, it has the `sample_weights` parameter for that

### 9.2 Using the same dataset from exercise 1,
#### 9.2 (a) Use RandomizedSearchCV on 10-fold CV to find the C, gamma optimal hyper-parameters on an SVC with RBF kernel, where `param_distributions={'C':logUniform(a=1E-2,b= 1E2),'gamma':logUniform(a=1E-2,b=1E2)} n_iter=25` and neg_log_loss is the scoring function.

In [3]:
from afml.modelling import hyperparameter_tuning
importlib.reload(hyperparameter_tuning)
from afml.modelling.hyperparameter_tuning import log_uniform, tune_hyper_params

search = tune_hyper_params(
    X,
    y['bin'],
    y['t1'],
    pipe_estimator=SVC(kernel='rbf'),
    param_grid={'C': log_uniform(a=1e-2, b=1e2),'gamma': log_uniform(a=1e-2, b=1e2)},
    cv=10,
    search_type='random',
    n_jobs=-1,
    pct_embargo=0,
    n_iter=25,
    scoring='neg_log_loss',
)

/Users/yakirhadad/PycharmProjects/Advances-In-Financial-Machine-Learning/venv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py:982: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/yakirhadad/PycharmProjects/Advances-In-Financial-Machine-Learning/venv/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 971, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yakirhadad/PycharmProjects/Advances-In-Financial-Machine-Learning/venv/lib/python3.11/site-packages/sklearn/metrics/_scorer.py", line 279, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yakirhadad/PycharmProjects/Advances-In-Fin

KeyboardInterrupt: 

In [15]:
y

,bin,w,t1
1725915069117,0,0.0001,1725915069117
1725915129117,0,0.0001,1725915129117
1725915189117,0,0.0001,1725915189117
1725915249117,0,0.0001,1725915249117
1725915309117,0,0.0001,1725915309117
...,...,...,...
1726514769117,1,0.0001,1726514769117
1726514829117,1,0.0001,1726514829117
1726514889117,1,0.0001,1726514889117
1726514949117,1,0.0001,1726514949117
